ДЕТЕКЦИЯ ШТРИХ-КОДОВ

In [1]:
import cv2
import numpy as np
import pandas as pd

In [2]:
#разметка
df = pd.read_csv('annotations.tsv', sep=',')
print(df.head())
print(df.shape)

                                            filename  x_from  y_from  width  \
0  images/0b56af7e-386c-410a-8f46-74350f755d77--r...     216     474    409   
1  images/6e042372-20ec-4a26-8340-3653cf0c681d--r...      23     849    230   
2  images/788fb7f3-e813-4754-874f-3cc6eb19113d--r...     322     632    344   
3  images/d05d50c0-0c19-4ca4-88ff-6f7058318a7f--r...     516     694    325   
4  images/93cda132-8b66-476a-a112-8acf47658fee--r...     200     474    531   

   height  
0     308  
1     125  
2     145  
3     156  
4     292  
(76, 5)


In [3]:
row = df.iloc[0]
filename = row['filename']
image = cv2.imread(filename)

print(filename)
print(image.shape)

images/0b56af7e-386c-410a-8f46-74350f755d77--ru.4c7208d1-cba4-4539-967d-1c87ad0f6d2e.jpg
(1333, 1000, 3)


отображение эталонной разметки

In [4]:
x = int(row['x_from'])
y = int(row['y_from'])
w = int(row['width'])
h = int(row['height'])

gt_image = cv2.imread(filename)
cv2.rectangle(gt_image, (x, y), (x + w, y + h), (0, 255, 0), 3)

cv2.imwrite('ground_truth.jpg', gt_image)

True

перевод фотки в градации серого

In [5]:
gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
print(gray.shape)
cv2.imwrite('gray.jpg', gray)

(1333, 1000)


True

black top_hat и выделение вертикальных границ (градиент)

In [6]:
blackhat_kernel = cv2.getStructuringElement(
    cv2.MORPH_RECT,
    (15, 9)
)

blackhat = cv2.morphologyEx(
    gray,
    cv2.MORPH_BLACKHAT,
    blackhat_kernel
)

sobel_x = np.array([
    [-1, 0, 1],
    [-2, 0, 2],
    [-1, 0, 1]
])

gradient = cv2.filter2D(blackhat, -1, sobel_x)

cv2.imwrite('gradient.jpg', gradient)

True

сглаживание и бинаризация

In [7]:
blurred = cv2.GaussianBlur(gradient, (5, 5), 0)

threshold_value, binary = cv2.threshold(
    blurred,
    0, #Otsu сам подбирает порог, поэтому исходное значение здесь можно поставить 0
    255,
    cv2.THRESH_BINARY + cv2.THRESH_OTSU
)

print(threshold_value)

cv2.imwrite('binary.jpg', binary)

67.0


True

объединение полос

In [8]:
kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (25, 3))
closed = cv2.morphologyEx(binary, cv2.MORPH_CLOSE, kernel)
cv2.imwrite('closed.jpg', closed)

True

поиск контуров

In [9]:
contours, hierarchy = cv2.findContours(
    closed,
    cv2.RETR_EXTERNAL, #только внешние контуры
    cv2.CHAIN_APPROX_SIMPLE #настройка: не сохраняй лишние промежуточные точки на прямых участках контура
)

contours = sorted(contours, key=cv2.contourArea, reverse=True)

print('Количество контуров:', len(contours))

Количество контуров: 109


In [10]:
contours_image = cv2.imread(filename)

for contour in contours[:5]:
    x_contour, y_contour, w_contour, h_contour = cv2.boundingRect(contour)
#баундингрект строит прямоуг вокруг контура и возвращ х у шир длин
    cv2.rectangle(
        contours_image,
        (x_contour, y_contour),
        (x_contour + w_contour, y_contour + h_contour),
        (0, 255, 0),
        3
    )

cv2.imwrite('contours.jpg', contours_image)

True

выбор найденной области

In [11]:
predicted_cont = contours[0]
x_pred, y_pred, w_pred, h_pred = cv2.boundingRect(predicted_cont)
result_image = cv2.imread(filename)

cv2.rectangle(result_image, (x, y), (x + w, y + h), (0, 255, 0), 3)

cv2.rectangle(result_image, (x_pred, y_pred), (x_pred + w_pred, y_pred + h_pred), (0, 0, 255), 3)

cv2.imwrite('prediction.jpg', result_image)

True

расчет IoU

In [12]:
x_left = max(x, x_pred)
y_top = max(y, y_pred)

x_right = min(x + w, x_pred + w_pred)
y_bottom = min(y + h, y_pred + h_pred)

intersection_width = max(0, x_right - x_left)
intersection_height = max(0, y_bottom - y_top)

intersection_area = intersection_width * intersection_height

gt_area = w * h
pred_area = w_pred * h_pred

union_area = gt_area + pred_area - intersection_area

iou = intersection_area / union_area

print('IoU:', iou)

IoU: 0.8868637474994443


запуск алгоритма на всех фотках

In [13]:
iou_values = []

for i in range(len(df)):
    current_row = df.iloc[i]

    current_filename = current_row['filename']

    current_image = cv2.imread(current_filename)

    current_gray = cv2.cvtColor(
    current_image,
    cv2.COLOR_BGR2GRAY
    )

    current_blackhat = cv2.morphologyEx(
        current_gray,
        cv2.MORPH_BLACKHAT,
        blackhat_kernel
    )

    current_gradient = cv2.filter2D(
        current_blackhat,
        -1,
        sobel_x
    )

    current_blurred = cv2.GaussianBlur(
        current_gradient,
        (5, 5),
        0
    )

    threshold_value, current_binary = cv2.threshold(
        current_blurred,
        0,
        255,
        cv2.THRESH_BINARY + cv2.THRESH_OTSU
    )

    current_closed = cv2.morphologyEx(
        current_binary,
        cv2.MORPH_CLOSE,
        kernel
    )

    current_contours, hierarchy = cv2.findContours(
        current_closed,
        cv2.RETR_EXTERNAL,
        cv2.CHAIN_APPROX_SIMPLE
    )

    current_contours = sorted(
        current_contours,
        key=cv2.contourArea,
        reverse=True
    )

    if len(current_contours) == 0:
        iou_current = 0

    else:
        predicted_contour = current_contours[0]

        x_pred, y_pred, w_pred, h_pred = cv2.boundingRect(
            predicted_contour
        )

        x_gt = int(current_row['x_from'])
        y_gt = int(current_row['y_from'])
        w_gt = int(current_row['width'])
        h_gt = int(current_row['height'])

        x_left = max(x_gt, x_pred)
        y_top = max(y_gt, y_pred)

        x_right = min(
            x_gt + w_gt,
            x_pred + w_pred
        )

        y_bottom = min(
            y_gt + h_gt,
            y_pred + h_pred
        )

        intersection_width = max(
            0,
            x_right - x_left
        )

        intersection_height = max(
            0,
            y_bottom - y_top
        )

        intersection_area = (
            intersection_width
            * intersection_height
        )

        gt_area = w_gt * h_gt
        pred_area = w_pred * h_pred

        union_area = (
            gt_area
            + pred_area
            - intersection_area
        )

        iou_current = intersection_area / union_area

    iou_values.append(iou_current)

    print(i, iou_current)

0 0.8868637474994443
1 0.8485565217391304
2 0.9750734583604488
3 0.9505116171718726
4 0.9319131646157418
5 0.9509602834234571
6 0.8603756758263196
7 0.9807032520979978
8 0.8557744937055282
9 0.7817982827291293
10 0.9065871209791501
11 0.9339865371652093
12 0.9200932890411194
13 0.636217168121698
14 0.8799073194951351
15 0.8889243451668661
16 0.9089007092198581
17 0.8551594914120009
18 0.8975064967207029
19 0.9636163175303197
20 0.9750566893424036
21 0.9097021448261952
22 0.5429231723127219
23 0.9278183292781833
24 0.9108770466159141
25 0.8718339774505733
26 0.9701046902755206
27 0.9021260997067448
28 0.7407075950122308
29 0.9516138655211502
30 0.8736747214896938
31 0.8729887024991442
32 0.9217727208877652
33 0.0
34 0.9100692327007928
35 0.8720138179341216
36 0.8577206045295395
37 0.0
38 0.2223159460513546
39 0.9437862950058072
40 0.8813611261434542
41 0.9482147985067484
42 0.9385178192117064
43 0.9101220457187137
44 0.8987393249288329
45 0.9525030919988583
46 0.8932442159383034
47 0.83

In [14]:
print('Средний IoU:', np.mean(iou_values))

Средний IoU: 0.8532358065484701


результат: сначала среднее было меньше 0.8, тк штрих-код я отбираю по принципу самого большого по площади контура, но не всегда именно он был самым крупным из-зи объединения текста с упаковки и тд, поэтому добавила black top-hat перед sobel. после этого собель лучше выделял именно вертикальные полосы